# Dashboard review briefing integration v1

目标：验证 `src/analysis` 产出的 `model_review_briefing` 和 `segment_effect_matrix` 可以通过 manifest/export 进入 `src/dashboard` 页面，并进入 `src/schedule` 的详细通知摘要。

In [1]:
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path.cwd().resolve()
while not ((REPO_ROOT / 'AGENTS.md').exists() and (REPO_ROOT / 'src').exists()):
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError('cannot locate repo root')
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.dashboard import (
    DashboardViewFilters,
    build_dashboard_view_model,
    load_model_matrix_dashboard,
    write_dashboard_html,
    write_model_matrix_dashboard_export,
)
from src.schedule import ModelDashboardPipelineConfig, run_model_dashboard_refresh_once

CARD = REPO_ROOT / 'research' / 'dashboard_review_briefing_integration_v1'
SOURCE_CARD = REPO_ROOT / 'research' / 'analysis_segment_effect_matrix_v1'
DASHBOARD_EXPORT = CARD / 'dashboard_export.json'
VIEW_MODEL = CARD / 'view_model_weakness.json'
HTML_PAGE = CARD / 'dashboard_review_briefing.html'
NOTIFICATION_EVENTS = CARD / 'notification_events.json'


In [2]:
dashboard_payload = write_model_matrix_dashboard_export(
    SOURCE_CARD,
    output_path=DASHBOARD_EXPORT,
    top_n=10,
)
view_model = build_dashboard_view_model(
    dashboard_payload,
    filters=DashboardViewFilters(review_signals=('weakness',), decision_hints=('block_or_rework',)),
)
html = write_dashboard_html(view_model, output_path=HTML_PAGE, title='SOXL Review Briefing Dashboard')
VIEW_MODEL.write_text(json.dumps(view_model, ensure_ascii=False, indent=2), encoding='utf-8')

assert dashboard_payload['model_review_briefing']
assert dashboard_payload['segment_effect_matrix']
assert view_model['controls']['review_signals']
assert view_model['filters_applied']['review_signals'] == ['weakness']
assert 'review-signal-filter' in html
assert 'Model Review Briefing' in html

{
    'dashboard_review_rows': len(dashboard_payload['model_review_briefing']),
    'segment_effect_rows': len(dashboard_payload['segment_effect_matrix']),
    'filtered_model_cards': len(view_model['model_cards']),
    'top_review_signal': view_model['review_briefing_summary']['top_review_signal'],
}


{'dashboard_review_rows': 5,
 'segment_effect_rows': 10,
 'filtered_model_cards': 1,
 'top_review_signal': 'weakness'}

In [3]:
events = []

def matrix_runner(**kwargs):
    return {'variant_count': len(dashboard_payload.get('top_models', [])), 'status': 'reuse_existing_analysis_card'}

def dashboard_exporter(card_dir: Path, *, output_path: Path, top_n: int):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(dashboard_payload, ensure_ascii=False, indent=2), encoding='utf-8')
    return dashboard_payload

config = ModelDashboardPipelineConfig(
    name='dashboard_review_briefing_refresh',
    card_dir=SOURCE_CARD,
    dashboard_output_path=CARD / 'scheduled_dashboard_export.json',
    start='2026-01-01',
    end='2026-01-08',
    train_end='2026-01-04',
    model_names=('ridge_mse', 'gcn_mse'),
    notify_details=True,
)
result = run_model_dashboard_refresh_once(
    config,
    lock_dir=CARD / 'locks',
    notify=events.append,
    matrix_runner=matrix_runner,
    dashboard_exporter=dashboard_exporter,
)
NOTIFICATION_EVENTS.write_text(json.dumps(events, ensure_ascii=False, indent=2), encoding='utf-8')

assert result.status == 'success'
assert result.payload['top_review_signal']
assert 'top_review=' in events[0]
{
    'schedule_status': result.status,
    'top_review_signal': result.payload['top_review_signal'],
    'top_review_segment': result.payload['top_review_segment'],
    'notification': events[0],
}


{'schedule_status': 'success',
 'top_review_signal': 'weakness',
 'top_review_segment': 'stress',
 'notification': 'dashboard_review_briefing_refresh:success | top_model=ridge_mse | dashboard_manifest=analysis_manifest | attempts=1 | export=/Users/rziiiii/dev/workspace/research/dashboard_review_briefing_integration_v1/scheduled_dashboard_export.json | top_failure=return_stability_lag@stress | top_review=weakness:block_or_rework@stress'}

In [4]:
summary = {
    'schema_version': 'research.dashboard_review_briefing_integration_v1',
    'status': 'completed',
    'source_card': str(SOURCE_CARD.relative_to(REPO_ROOT)),
    'dashboard_export': str(DASHBOARD_EXPORT.relative_to(REPO_ROOT)),
    'view_model': str(VIEW_MODEL.relative_to(REPO_ROOT)),
    'html_page': str(HTML_PAGE.relative_to(REPO_ROOT)),
    'notification_events': str(NOTIFICATION_EVENTS.relative_to(REPO_ROOT)),
    'dashboard_review_rows': len(dashboard_payload['model_review_briefing']),
    'segment_effect_rows': len(dashboard_payload['segment_effect_matrix']),
    'filtered_model_card_count': len(view_model['model_cards']),
    'review_signals': view_model['controls']['review_signals'],
    'decision_hints': view_model['controls']['decision_hints'],
    'top_review_signal': result.payload['top_review_signal'],
    'top_review_decision_hint': result.payload['top_review_decision_hint'],
    'top_review_segment': result.payload['top_review_segment'],
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary


{'schema_version': 'research.dashboard_review_briefing_integration_v1',
 'status': 'completed',
 'source_card': 'research/analysis_segment_effect_matrix_v1',
 'dashboard_export': 'research/dashboard_review_briefing_integration_v1/dashboard_export.json',
 'view_model': 'research/dashboard_review_briefing_integration_v1/view_model_weakness.json',
 'html_page': 'research/dashboard_review_briefing_integration_v1/dashboard_review_briefing.html',
 'notification_events': 'research/dashboard_review_briefing_integration_v1/notification_events.json',
 'dashboard_review_rows': 5,
 'segment_effect_rows': 10,
 'filtered_model_card_count': 1,
 'review_signals': ['strength', 'weakness'],
 'decision_hints': ['block_or_rework', 'promote_candidate'],
 'top_review_signal': 'weakness',
 'top_review_decision_hint': 'block_or_rework',
 'top_review_segment': 'stress'}

## 中文结论

- 做了什么：验证 `model_review_briefing` 与 `segment_effect_matrix` 已进入 dashboard manifest/export、view model、静态 HTML 和 schedule 详细通知链路。
- 核心指标：dashboard export 读取到 review briefing 与 segment effect rows；按 `weakness/block_or_rework` 过滤后仍能生成模型卡片；schedule 通知包含 `top_review=` 摘要。
- regime 差异：最高优先级 review 指向 stress/window_regime 弱点，页面和通知可以直接定位需重做或阻断的模型分段。
- 是否晋升：晋升为 dashboard/schedule 框架能力；不代表任何交易模型晋升 production。
- 风险：本轮复用 `analysis_segment_effect_matrix_v1` 的样例数据；真实 QuantBench 全量结果需要后续接入同一 export。
- 下一步：让 dashboard 页面增加 review_signal/decision_hint 的视觉回归检查，并把通知接到真实定时任务配置。